In [ ]:
from pathlib import Path
import yaml

In [3]:
def get_organ_task (organ_yaml: str):
    with open(organ_yaml,"r") as f:
        organs = yaml.safe_load(f)
        
    organs_in_total = []
    organs_in_head = []
    all_organs = []
    tasks = set()
    
    for organ in organs["organs"]:
        all_organs.append(organ["source_class"]) #get all required organ 
        tasks.add(organ["task"])
        
        if (organ["task"]) == "total":
            organs_in_total.append(organ["source_class"]) #get organ in total (for roi_subset)
        else:
            organs_in_head.append(organ["source_class"]) #get organ in head_glands_cavities (for masks selection)
            
    return all_organs, organs_in_total, organs_in_head, tasks   
    #list of all organ, organ in total (for roi_subset), organ in head_glands_cavities (for masks selection)

In [4]:

# get the gender if a specific subject form gender.yaml
# subject_name: is the subject name in quadra dataset eg: QUADRA_HC_021
def get_gender (subject_name: str):   
    with open("/workspaces/segmentation/gender.yaml", "r") as f: #loading the yaml file
        genders = yaml.safe_load(f)
        
        for subject in genders: #get dict{subject: gender} 
            if subject_name in subject:
                gender = subject[subject_name] #get the gender
                
    return gender

get_gender("QUADRA_HC_021")

'F'

In [5]:
#obtaining all mask dirs in output_folder
def get_mask_dirs  (output_folder: str):
    output_dir = Path(output_folder)
    mask_dirs = {}

    #get organ list for male and female (without prostate)
    all_organ, _, _, _ = get_organ_task ("/workspaces/segmentation/organs.yaml")
    female_organ = [ organ for organ in all_organ if organ != "prostate"] #excluding prostate
    required_organ = {"M": all_organ, "F": female_organ} # put it in a dict to prevent nesting if later
    
    for subject_dir in sorted(output_dir.iterdir()): #iterate through all subjects
        subject_name = subject_dir.name
        mask_per_subject = {}
        gender = get_gender(subject_name)
        organ_for_gender = required_organ[gender] #selecting organ list based on gender
        
        for session_dir in sorted(subject_dir.iterdir()): #iterate thorugh all sessions
            session_name = session_dir.name
            mask_per_session = []
            
            for mask_dir in sorted(session_dir.iterdir()): #iterate through all masks
                mask_name = mask_dir.name.removesuffix(".nii.gz")

                if mask_name in organ_for_gender:
                    mask_per_session.append(mask_dir) # list of mask dirs based on organ list for a specific gender

            mask_per_subject[session_name] = mask_per_session # assign the mask dirs list to the session
        
        mask_dirs[subject_name] = mask_per_subject # assign all mask in the session above to the subject

    return mask_dirs #dict{subject: {session: [mask_dirs]} }


In [9]:
# using the selected maks dirs from get_mask_dirs copy the masks form output_folder to selected_mask_folder
def selecting_masks (selected_mask_folder: str, output_folder:str):

    selected_mask_dir = Path(selected_mask_folder)
    mask_dirs = get_mask_dirs(output_folder) #get the source dirs
    
    for subject in mask_dirs:

        for session in mask_dirs[subject]:
            target_dir = selected_mask_dir / subject / session #formulating the destination dirs
            target_dir.mkdir(parents=True, exist_ok=True) #creating the destination dirs
            if not any(target_dir.iterdir()):
                for mask in mask_dirs[subject][session]:
                    mask.copy_into(target_dir, preserve_metadata=True) #copying the masks form source to destination dirs
            else:
                raise ValueError (f"masks are existed in folder: selected_mask '{subject}'/'{session}'")

In [ ]:
selecting_masks ("..data/output", "..data/transit_folder")

ValueError: masks are existed in folder: selected_mask 'QUADRA_HC_021'/'retest'

In [8]:
selected_mask_dir = get_mask_dirs("../selected_mask")
selected_mask_dir

{'QUADRA_HC_021': {'retest': [PosixPath('../selected_mask/QUADRA_HC_021/retest/brain.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/colon.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/duodenum.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/esophagus.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/eye_left.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/eye_right.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/heart.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/hip_left.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/hip_right.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/kidney_left.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/kidney_right.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/liver.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC_021/retest/lung_lower_lobe_left.nii.gz'),
   PosixPath('../selected_mask/QUADRA_HC